## Basic RAG

1.Install langchain

In [2]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf openai -q

In [3]:
import os
import openai

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from chromadb.config import Settings

/tmp/ipykernel_1136/2720623074.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


2.Upload the document based on domain(Travel Tourism)

In [4]:
import os
from google.colab import files

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
    UnstructuredWordDocumentLoader
)

# Upload file
uploaded = files.upload()

file_path = list(uploaded.keys())[0]

print("Uploaded File:", file_path)

# Detect extension
ext = os.path.splitext(file_path)[1].lower()

# Select loader
if ext == ".pdf":
    loader = PyPDFLoader(file_path)

elif ext == ".txt":
    loader = TextLoader(file_path)

elif ext == ".csv":
    loader = CSVLoader(file_path)

elif ext == ".docx":
    loader = UnstructuredWordDocumentLoader(file_path)

else:
    raise ValueError(f"Unsupported file type: {ext}")

# Load documents
documents = loader.load()

print("Documents Loaded:", len(documents))
print(documents[0].page_content[:1000])

Saving Travel_Tourism_RAG_Knowledge_Base.txt to Travel_Tourism_RAG_Knowledge_Base.txt
Uploaded File: Travel_Tourism_RAG_Knowledge_Base.txt
Documents Loaded: 1
TRAVEL & TOURISM RAG KNOWLEDGE BASE
Version: 1.0
Purpose: General-purpose knowledge base for a Travel & Tourism RAG chatbot.

1. TRAVEL PLANNING
Travel planning involves selecting a destination, deciding travel dates, estimating a budget, arranging transport and accommodation, planning activities, and checking safety or entry requirements.
A useful itinerary normally contains destination, date, transport, accommodation, activities, meal breaks, estimated costs, and free time.
Travelers should verify prices, operating hours, weather, entry requirements, and local rules before a trip because these can change.

2. TYPES OF TOURISM
Leisure tourism: Travel mainly for relaxation, recreation, sightseeing, beaches, resorts, or entertainment.
Adventure tourism: Activities such as trekking, rafting, camping, rock climbing, diving, and wild

3.Distribution of chunks

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))

Total Chunks: 22


In [6]:
from google.colab import userdata
api_key=userdata.get('api_key')
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://nexusapi.navigatelabs.ai"
)

In [7]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="rag"
)

Embedding the code

In [8]:
for i, doc in enumerate(docs):

    text = doc.page_content

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    collection.add(
        documents=[text],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("Embeddings stored successfully!")

Embeddings stored successfully!


Add Query(Text + Image)

In [15]:
from IPython.display import display
from ipywidgets import FileUpload, Text, Button, Output
from PIL import Image
import io

# Question input
query_box = Text(
    description="Question:",
    placeholder="Ask something about the place..."
)

# Image upload
image_upload = FileUpload(
    accept="image/*",
    multiple=False
)

# Submit button
submit_button = Button(
    description="Ask",
    button_style="success"
)

output = Output()

display(query_box)
display(image_upload)
display(submit_button)
display(output)


def ask_question(b):
    with output:
        output.clear_output()

        query = query_box.value

        print("Question:", query)

        # Check if an image was uploaded
        if image_upload.value:
            print("Image uploaded successfully!")

            # Get uploaded image
            uploaded_file = list(image_upload.value.values())[0]
            image_data = uploaded_file["content"]

            # Open image
            image = Image.open(io.BytesIO(image_data))

            display(image)

        else:
            print("No image uploaded.")

        # Your RAG code goes here
        # response = rag_chain.invoke(query)
        # print(response)


submit_button.on_click(ask_question)

Text(value='', description='Question:', placeholder='Ask something about the place...')

FileUpload(value={}, accept='image/*', description='Upload')

Button(button_style='success', description='Ask', style=ButtonStyle())

Output()

In [18]:
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_embedding = query_response.data[0].embedding

In [19]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

In [20]:
retrieved_chunks = results["documents"][0]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\nChunk {i+1}")
    print("-" * 50)
    print(chunk)


Chunk 1
--------------------------------------------------
3. POPULAR INDIAN DESTINATIONS
Delhi: Known for historical monuments, museums, markets, and major landmarks including India Gate, Red Fort, Qutub Minar, and Humayun's Tomb.
Agra: Known for the Taj Mahal, Agra Fort, and Mughal heritage.
Jaipur: Known for forts, palaces, markets, and Rajasthani culture. Major attractions include Amber Fort, City Palace, and Hawa Mahal.
Goa: Known for beaches, coastal scenery, Portuguese-influenced heritage, water activities, and nightlife.

Chunk 2
--------------------------------------------------
Chikkamagaluru: Known for coffee estates, hills, waterfalls, trekking, and scenic landscapes.
Gokarna: Coastal destination known for beaches and temples.
Kabini: Known for wildlife tourism, forest landscapes, and safari experiences.

Chunk 3
--------------------------------------------------
Kerala: Known for backwaters, hill stations, beaches, Ayurveda and wellness tourism, and cultural experiences.


In [21]:
# 1. Get the retrieved chunks
retrieved_chunks = results["documents"][0]

# 2. Combine all retrieved chunks into one context
context = "\n\n".join(retrieved_chunks)

# 3. Check the context
print(context)

3. POPULAR INDIAN DESTINATIONS
Delhi: Known for historical monuments, museums, markets, and major landmarks including India Gate, Red Fort, Qutub Minar, and Humayun's Tomb.
Agra: Known for the Taj Mahal, Agra Fort, and Mughal heritage.
Jaipur: Known for forts, palaces, markets, and Rajasthani culture. Major attractions include Amber Fort, City Palace, and Hawa Mahal.
Goa: Known for beaches, coastal scenery, Portuguese-influenced heritage, water activities, and nightlife.

Chikkamagaluru: Known for coffee estates, hills, waterfalls, trekking, and scenic landscapes.
Gokarna: Coastal destination known for beaches and temples.
Kabini: Known for wildlife tourism, forest landscapes, and safari experiences.

Kerala: Known for backwaters, hill stations, beaches, Ayurveda and wellness tourism, and cultural experiences.
Karnataka: Known for Bengaluru, Mysuru, Hampi, Coorg, Chikkamagaluru, Gokarna, and diverse heritage and natural attractions.
Rajasthan: Known for forts, palaces, deserts, heritag

In [23]:
prompt = f"""
Answer the question using the context below. If there is no context, you can answer on your own

Context:
{context}

Question:
{query}
"""

In [24]:
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print("\nFINAL ANSWER")
print("=" * 50)

print(answer)


FINAL ANSWER
With a budget of under 20,000 INR, you can still enjoy several popular destinations in India by choosing affordable accommodation, traveling during off-peak seasons, and opting for economical transport options. Some good options include:

1. Delhi: Explore historical monuments like India Gate, Red Fort, Qutub Minar, Humayun's Tomb, and local markets. These sites have minimal entry fees, and city transportation can be managed via metro and buses.

2. Agra: Visit the Taj Mahal, Agra Fort, and Mughal heritage sites. Transportation and entry fees are economical, making it suitable under your budget.

3. Jaipur: Discover forts, palaces, markets, and Rajasthani culture. Attractions like Amber Fort and Hawa Mahal can be visited with reasonable entry fees, and local transport options are affordable.

4. Gokarna: Enjoy beaches and temples without high costs, especially if you choose budget accommodations and local eateries.

5. Kerala (Backwaters and Hill Stations): Opt for budget

In [25]:
while True:

    query = input("\nAsk your travel question: ")

    # Exit immediately if user wants to quit
    if query.lower().strip() in ["exit", "quit"]:
        print("\nThank you for using the Travel & Tourism Assistant!")
        break

    # Query embedding
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    query_embedding = query_response.data[0].embedding

    # Retrieve relevant documents
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_chunks = results["documents"][0]

    # Combine retrieved chunks
    context = "\n\n".join(retrieved_chunks)

    # Prompt
    prompt = f"""
You are a helpful Travel & Tourism Assistant.

Answer the user's question using the provided context.

You can also provide practical travel suggestions when appropriate.

If the user asks for recommendations or better options:
- Compare the available options clearly.
- Consider budget, travel experience, activities, transport and accommodation.
- If spending around ₹5,000 more could provide a significantly better option,
  mention that option and explain what additional benefit the user may get.
- Do not force the user to spend more.
- If the context does not contain enough information, clearly say that the
  available travel knowledge base does not provide enough information.

Context:
{context}

User Question:
{query}

Give a clear, useful and easy-to-understand answer.
"""

    # LLM response
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    print("\nTravel Assistant:")
    print("-" * 50)
    print(answer)
    print("-" * 50)

    # Ask whether user wants to continue
    while True:

        choice = input(
            "\nDo you want to ask another question? (yes/no): "
        ).lower().strip()

        if choice in ["yes", "y"]:
            break

        elif choice in ["no", "n", "exit", "quit"]:
            print("\nThank you for using the Travel & Tourism Assistant!")
            raise SystemExit

        else:
            print("Please enter yes or no.")


Ask your travel question: I have ₹20,000 for a 3-day trip. What are my options?

Travel Assistant:
--------------------------------------------------
With a budget of ₹20,000 for a 3-day trip, you have several options depending on the destination, type of experience you want, and your preferred mode of travel. Here's a simple overview:

1. **Local or Nearby Destinations in India:**
   - **Budget:** Many popular hill stations, beaches, or cultural cities can be explored comfortably within this budget.
   - **Transport:** Use trains or buses for economical travel. For convenience, you could consider shared taxis or app-based cab services for local transport.
   - **Accommodation:** Budget hotels, hostels, or homestays can cost around ₹1,000-₹2,000 per night.
   - **Activities & Food:** Local eateries and free or low-cost attractions help keep costs low.
   
2. **Sample Itinerary (e.g., a nearby hill station or coastal town):**
   - **Transport:** Book trains or buses well in advance for

SystemExit: 

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
